# Week 4 – Databricks ETL for Smart Energy Monitoring
**Capstone: Smart Home Energy Usage Tracker**

## Setup

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.appName("SmartEnergyETL").getOrCreate()
print("Spark Session Created")

Spark Session Created


## Load Dataset

In [2]:
from google.colab import files
uploaded = files.upload()

Saving energy_usage.csv to energy_usage.csv


In [4]:
df = spark.read.csv("energy_usage.csv", header=True, inferSchema=True)
df = df.withColumn("timestamp", to_timestamp(col("timestamp")))
df = df.withColumn("date",  to_date(col("timestamp")))
df = df.withColumn("week",  weekofyear(col("timestamp")))
df = df.withColumn("month", date_format(col("timestamp"), "yyyy-MM"))

print("Schema:")
df.printSchema()

print("Raw Data:")
df.show()

Schema:
root
 |-- log_id: integer (nullable = true)
 |-- device_id: string (nullable = true)
 |-- device_name: string (nullable = true)
 |-- room_id: string (nullable = true)
 |-- room_name: string (nullable = true)
 |-- energy_kwh: double (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- status: string (nullable = true)
 |-- date: date (nullable = true)
 |-- week: integer (nullable = true)
 |-- month: string (nullable = true)

Raw Data:
+------+---------+---------------+-------+------------+----------+-------------------+------+----------+----+-------+
|log_id|device_id|    device_name|room_id|   room_name|energy_kwh|          timestamp|status|      date|week|  month|
+------+---------+---------------+-------+------------+----------+-------------------+------+----------+----+-------+
|     1|     D001|Air Conditioner|   R001| Living Room|      1.82|2025-04-01 00:00:00|active|2025-04-01|  14|2025-04|
|     2|     D002|   Refrigerator|   R002|     Kitchen|      0.45|202

In [5]:
# Register the cleaned DataFrame as a temp view
df.createOrReplaceTempView("energy_logs")
print("Temp view 'energy_logs' registered")

Temp view 'energy_logs' registered


## Daily Summary

In [6]:
daily_summary = spark.sql("""
    SELECT
        device_id,
        device_name,
        room_name,
        date,
        ROUND(SUM(energy_kwh), 3) AS daily_kwh,
        COUNT(*)                  AS num_readings
    FROM energy_logs
    GROUP BY device_id, device_name, room_name, date
    ORDER BY date, device_name
""")

daily_summary.createOrReplaceTempView("daily_summary")
print("Daily Summary")
daily_summary.show()

Daily Summary
+---------+---------------+------------+----------+---------+------------+
|device_id|    device_name|   room_name|      date|daily_kwh|num_readings|
+---------+---------------+------------+----------+---------+------------+
|     D001|Air Conditioner| Living Room|2025-04-01|     3.72|           2|
|     D009|    Ceiling Fan| Living Room|2025-04-01|     0.14|           2|
|     D007|     Dishwasher|     Kitchen|2025-04-01|     1.35|           2|
|     D008| Laptop Charger|     Bedroom|2025-04-01|     0.11|           2|
|     D005|      Microwave|     Kitchen|2025-04-01|     0.19|           2|
|     D002|   Refrigerator|     Kitchen|2025-04-01|     0.89|           2|
|     D010|    Room Heater|     Bedroom|2025-04-01|     2.95|           2|
|     D004|     Television| Living Room|2025-04-01|     0.27|           2|
|     D003|Washing Machine|Utility Room|2025-04-01|     1.78|           2|
|     D006|   Water Heater|    Bathroom|2025-04-01|      2.3|           2|
|     D001|

## Weekly Summary

In [7]:
weekly_summary = spark.sql("""
    SELECT
        device_id,
        device_name,
        week,
        ROUND(SUM(energy_kwh), 3) AS weekly_kwh
    FROM energy_logs
    GROUP BY device_id, device_name, week
    ORDER BY week, device_name
""")

weekly_summary.createOrReplaceTempView("weekly_summary")
print("Weekly Summary")
weekly_summary.show()

Weekly Summary
+---------+---------------+----+----------+
|device_id|    device_name|week|weekly_kwh|
+---------+---------------+----+----------+
|     D001|Air Conditioner|  14|      19.5|
|     D009|    Ceiling Fan|  14|       0.5|
|     D007|     Dishwasher|  14|      4.11|
|     D008| Laptop Charger|  14|      0.35|
|     D005|      Microwave|  14|      0.69|
|     D002|   Refrigerator|  14|      4.03|
|     D010|    Room Heater|  14|     10.68|
|     D004|     Television|  14|      1.08|
|     D003|Washing Machine|  14|      5.43|
|     D006|   Water Heater|  14|      9.68|
+---------+---------------+----+----------+



## Usage Alerts

In [8]:
alerts_df = spark.sql("""
    SELECT
        *,
        CASE
            WHEN daily_kwh > 10.0 THEN 'HIGH USAGE ALERT'
            ELSE 'NORMAL'
        END AS usage_alert
    FROM daily_summary
""")

alerts_df.createOrReplaceTempView("usage_alerts")
print("Usage Alerts")
alerts_df.show()

Usage Alerts
+---------+---------------+------------+----------+---------+------------+-----------+
|device_id|    device_name|   room_name|      date|daily_kwh|num_readings|usage_alert|
+---------+---------------+------------+----------+---------+------------+-----------+
|     D001|Air Conditioner| Living Room|2025-04-01|     3.72|           2|     NORMAL|
|     D009|    Ceiling Fan| Living Room|2025-04-01|     0.14|           2|     NORMAL|
|     D007|     Dishwasher|     Kitchen|2025-04-01|     1.35|           2|     NORMAL|
|     D008| Laptop Charger|     Bedroom|2025-04-01|     0.11|           2|     NORMAL|
|     D005|      Microwave|     Kitchen|2025-04-01|     0.19|           2|     NORMAL|
|     D002|   Refrigerator|     Kitchen|2025-04-01|     0.89|           2|     NORMAL|
|     D010|    Room Heater|     Bedroom|2025-04-01|     2.95|           2|     NORMAL|
|     D004|     Television| Living Room|2025-04-01|     0.27|           2|     NORMAL|
|     D003|Washing Machine|Uti

## Final Device Report

In [9]:
final_df = spark.sql("""
    SELECT
        device_name,
        ROUND(SUM(daily_kwh), 3)  AS total_kwh,
        ROUND(AVG(daily_kwh), 3)  AS avg_daily_kwh,
        COUNT(date)               AS days_logged
    FROM daily_summary
    GROUP BY device_name
    ORDER BY total_kwh DESC
""")

final_df.createOrReplaceTempView("device_report")
print("Final Device Report")
final_df.show()

Final Device Report
+---------------+---------+-------------+-----------+
|    device_name|total_kwh|avg_daily_kwh|days_logged|
+---------------+---------+-------------+-----------+
|Air Conditioner|     19.5|          3.9|          5|
|    Room Heater|    10.68|        2.136|          5|
|   Water Heater|     9.68|        1.936|          5|
|Washing Machine|     5.43|        1.086|          5|
|     Dishwasher|     4.11|        0.822|          5|
|   Refrigerator|     4.03|        0.806|          5|
|     Television|     1.08|        0.216|          5|
|      Microwave|     0.69|        0.138|          5|
|    Ceiling Fan|      0.5|          0.1|          5|
| Laptop Charger|     0.35|         0.07|          5|
+---------------+---------+-------------+-----------+



## Energy Savings Opportunities (SQL)

Identify devices whose average daily usage is materially above the fleet average — prime candidates for scheduling, replacement, or behavioural changes.

In [10]:
savings_df = spark.sql("""
    WITH fleet_avg AS (
        SELECT ROUND(AVG(avg_daily_kwh), 3) AS fleet_avg_kwh
        FROM device_report
    )
    SELECT
        dr.device_name,
        dr.avg_daily_kwh,
        fa.fleet_avg_kwh,
        ROUND(dr.avg_daily_kwh - fa.fleet_avg_kwh, 3)          AS excess_kwh_per_day,
        ROUND((dr.avg_daily_kwh - fa.fleet_avg_kwh) * 30, 3)   AS potential_monthly_savings_kwh,
        CASE
            WHEN dr.avg_daily_kwh > fa.fleet_avg_kwh * 1.5 THEN 'HIGH SAVINGS POTENTIAL'
            WHEN dr.avg_daily_kwh > fa.fleet_avg_kwh        THEN 'MODERATE SAVINGS POTENTIAL'
            ELSE 'EFFICIENT'
        END AS savings_opportunity
    FROM device_report dr
    CROSS JOIN fleet_avg fa
    ORDER BY excess_kwh_per_day DESC
""")

print("Energy Savings Opportunities")
savings_df.show()

Energy Savings Opportunities
+---------------+-------------+-------------+------------------+-----------------------------+--------------------+
|    device_name|avg_daily_kwh|fleet_avg_kwh|excess_kwh_per_day|potential_monthly_savings_kwh| savings_opportunity|
+---------------+-------------+-------------+------------------+-----------------------------+--------------------+
|Air Conditioner|          3.9|        1.121|             2.779|                        83.37|HIGH SAVINGS POTE...|
|    Room Heater|        2.136|        1.121|             1.015|                        30.45|HIGH SAVINGS POTE...|
|   Water Heater|        1.936|        1.121|             0.815|                        24.45|HIGH SAVINGS POTE...|
|Washing Machine|        1.086|        1.121|            -0.035|                        -1.05|           EFFICIENT|
|     Dishwasher|        0.822|        1.121|            -0.299|                        -8.97|           EFFICIENT|
|   Refrigerator|        0.806|        1.12

## Save Outputs

In [11]:
final_df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("/FileStore/output/energy_summary_csv")
print("CSV Saved")

CSV Saved


In [ ]:
final_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/FileStore/output/energy_summary_delta")
print("Delta Table Saved")

In [13]:
savings_df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("/FileStore/output/energy_savings_opportunities_csv")
print("Savings Opportunities CSV Saved")

Savings Opportunities CSV Saved
